# 09 — Combined slice + surface figures

In [ ]:
import os
os.environ["PYVISTA_OFF_SCREEN"] = "true"
import sys; sys.path.insert(0, "utils")
from config import *

from textwrap import fill
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from nilearn import plotting
import yabplot as yab

# Conte69 midthickness surfaces + the four standard views (left/right x lateral/medial).
LH, RH = yab.data.get_surface_paths("midthickness", "bmesh")
VIEWS = ["left_lateral", "left_medial", "right_lateral", "right_medial"]

# Subcortical axial z-slices (mm) -- matches 04_plot_slices.
CUTS = [-26, -10, 6, 22]

brain, affine, BG = load_mask()
N_VOX = int(brain.sum())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("FIG_DIR     :", FIG_DIR)
print("axial cuts z =", CUTS, "| in-mask voxels =", N_VOX, "| THR =", round(float(THR), 3))

## Shared helpers

`proj` projects a PALM volume to the surface with the projection rule preserved (p-maps ->
nearest, continuous T -> linear). `autocrop` trims the near-white border. `surf_strip` renders
the four views as a **1x4 horizontal strip** (`layout=(1, 4)`) into a cropped screenshot.

In [ ]:
def proj(family, fname, key, interp="linear"):
    """Project a PALM output volume to the Conte69 midthickness surface.

    p-value maps must use interp='nearest' to keep the threshold boundary exact; the
    continuous signed T-statistic uses interp='linear'.
    """
    return yab.project_vol2surf(str(result_path(family, key, fname)),
                                mask_medial_wall=True, interpolation=interp)


def autocrop(img):
    """Trim the near-white border so brains fill the cell."""
    g = img[..., :3].mean(-1)
    m = g < 250
    if not m.any():
        return img
    ys, xs = np.where(m)
    return img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]


def surf_strip(lh_d, rh_d, cmap, vminmax):
    """Render the 4 views as a HORIZONTAL 1x4 strip (layout=(1, 4)) -> cropped array."""
    lh_m, rh_m = yab.load_vertexwise_mesh(LH, RH, lh_d, rh_d)
    pl = yab.plot_vertexwise(lh_m, rh_m, views=VIEWS, layout=(1, 4), cmap=cmap,
                             vminmax=vminmax, nan_color=(0.83, 0.83, 0.83),
                             display_type="object", zoom=1.5)
    try:
        for nm in list(pl.scalar_bars.keys()):
            pl.remove_scalar_bar(nm)
    except Exception:
        pass
    img = pl.screenshot(transparent_background=False, return_img=True, scale=3)
    pl.close()
    return autocrop(img)

# Figure functions

In [ ]:
def combined_figure(family, pmap, title, outname, thresholded=True, surface=True):
    """Combined axial-slice + cortical-surface figure (6 domain rows) for one PALM family.

    For each ready domain the SAME signed voxelwise T (F_TSTAT) is shown twice, masked by
    significance (pmap >= THR): once as subcortical axial slices, once as a 1x4 cortical
    surface strip. One shared symmetric robust T scale (97.5th pct of |F_TSTAT| over the
    ready domains, in-brain) and one shared bottom diverging colorbar. Domains with no
    surviving voxels show a single 'n.s.' centred across the row. Gated on has_results(family).
    """
    ready = [k for k in KEYS if has_results(family, k)]
    if not ready:
        print(f"PALM not finished for {family} -- skipping {outname}")
        return None

    # --- shared symmetric robust T scale (97.5th pct of |F_TSTAT| over ready domains) ------
    allT = np.abs(np.concatenate([
        load_map(family, k, F_TSTAT, brain=brain)[brain] for k in ready]))
    VMAX_T = float(np.round(np.nanpercentile(allT, 97.5)))
    if VMAX_T <= 0:
        VMAX_T = float(np.round(np.nanmax(allT))) or 1.0
    print(f"[{family}] {title}: shared |T| cap +/- {VMAX_T:.0f} "
          f"(max |T| in-brain {np.nanmax(allT):.1f}); {len(ready)}/{len(KEYS)} domains ready")

    # --- per-domain volumetric significance-masked signed-T images (for the slice cell) ----
    def sig_T_volume(k):
        T = load_map(family, k, F_TSTAT, brain=brain)
        if thresholded:
            p = load_map(family, k, pmap, brain=brain)      # -log10(p)
            data = np.where(p >= THR, T, 0.0).astype(np.float32)
        else:
            data = T.astype(np.float32)                     # unthresholded: full signed T
        return nib.Nifti1Image(data, affine), bool(np.any(data != 0))

    # --- per-domain surface signed-T masked by the projected p-map (for the surface cell) --
    def sig_T_surface(k):
        tl, tr = proj(family, F_TSTAT, k)                   # signed T (linear)
        if thresholded:
            pl_, pr = proj(family, pmap, k, "nearest")      # p-map (nearest)
            ml = np.where(pl_ >= THR, tl, np.nan)
            mr = np.where(pr >= THR, tr, np.nan)
        else:
            ml, mr = tl, tr                                  # unthresholded: full signed T
        has = bool(np.any(np.isfinite(np.concatenate([ml, mr]))))
        return (ml, mr), has

    nrow = len(ready)
    ncol = 3 if surface else 2
    wratios = [0.18, 1.00, 1.70] if surface else [0.18, 1.00]
    fig = plt.figure(figsize=((16.5 if surface else 8.5), 2.3 * nrow))
    # [domain label | axial slices | (optional) cortical surface strip]
    gs = fig.add_gridspec(nrow, ncol, width_ratios=wratios, wspace=0.02, hspace=0.06,
                          left=0.01, right=0.99, top=0.93, bottom=0.085)
    ax_slice = [fig.add_subplot(gs[i, 1]) for i in range(nrow)]
    ax_surf = [fig.add_subplot(gs[i, 2]) for i in range(nrow)] if surface else [None] * nrow
    fig.canvas.draw()      # finalize cell positions before drawing maps into them

    slice_disp = []        # nilearn display per row (None where n.s.)
    surf_has = []          # bool: surface cell had surviving voxels
    for i, k in enumerate(ready):
        # --- axial slices ---
        vimg, vhas = sig_T_volume(k)
        if vhas:
            d = plotting.plot_stat_map(
                vimg, bg_img=BG, display_mode="z", cut_coords=CUTS, axes=ax_slice[i],
                threshold=1e-6, vmax=VMAX_T, cmap="RdBu_r", colorbar=False,
                annotate=False, black_bg=False, radiological=True)
            slice_disp.append(d)
        else:
            ax_slice[i].axis("off")
            slice_disp.append(None)

        # --- cortical surface 1x4 strip (skipped when surface=False) ---
        if surface:
            (sl, sr), shas = sig_T_surface(k)
            ax_surf[i].axis("off")
            if shas:
                ax_surf[i].imshow(surf_strip(sl, sr, "RdBu_r", [-VMAX_T, VMAX_T]), aspect="equal")
            surf_has.append(shas)
        else:
            surf_has.append(True)

    # --- R/L markers on first cut of the TOP row only (radiological: R on viewer's left) ----
    if slice_disp[0] is not None:
        ax_first = list(slice_disp[0].axes.values())[0].ax
        ax_first.text(0.03, 0.97, "R", transform=ax_first.transAxes, ha="left", va="top",
                      fontsize=9, fontweight="bold", color="black")
        ax_first.text(0.97, 0.97, "L", transform=ax_first.transAxes, ha="right", va="top",
                      fontsize=9, fontweight="bold", color="black")

    # --- per-row geometry helpers --------------------------------------------------------
    def slice_ycenter(i):
        d = slice_disp[i]
        if d is not None:
            try:
                ps = [c.ax.get_position() for c in d.axes.values()]
                return (min(p.y0 for p in ps) + max(p.y1 for p in ps)) / 2
            except Exception:
                pass
        p = ax_slice[i].get_position(); return (p.y0 + p.y1) / 2

    # --- column headers above the top row ------------------------------------------------
    drawn = [d for d in slice_disp if d is not None]
    if drawn:
        ytop = max(c.ax.get_position().y1 for d in drawn for c in d.axes.values())
    else:
        ytop = gs[0, 1].get_position(fig).y1
    yhead = ytop + 0.010
    ps = gs[0, 1].get_position(fig); fig.text((ps.x0 + ps.x1) / 2, yhead, "Axial slices",
                                              fontsize=14, fontweight="bold", ha="center", va="bottom")
    if surface:
        pu = gs[0, 2].get_position(fig); fig.text((pu.x0 + pu.x1) / 2, yhead, "Cortical surface",
                                                  fontsize=14, fontweight="bold", ha="center", va="bottom")

    # --- domain labels + n.s. markers (slice cell AND surface cell) -----------------------
    x_lab = gs[0, 1].get_position(fig).x0 - 0.006
    slice_cx = (gs[0, 1].get_position(fig).x0 + gs[0, 1].get_position(fig).x1) / 2
    surf_cx = ((gs[0, 2].get_position(fig).x0 + gs[0, 2].get_position(fig).x1) / 2) if surface else slice_cx
    for i, k in enumerate(ready):
        yc = slice_ycenter(i)
        fig.text(x_lab, yc, fill(DOMAINS[k], 16), va="center", ha="right",
                 fontsize=12, fontweight="bold")
        if slice_disp[i] is None and not surf_has[i]:
            row_cx = (gs[0, 1].get_position(fig).x0 + gs[0, ncol-1].get_position(fig).x1) / 2
            fig.text(row_cx, yc, "n.s.", va="center", ha="center", style="italic", fontsize=12)
        elif slice_disp[i] is None:
            fig.text(slice_cx, yc, "n.s.", va="center", ha="center", style="italic", fontsize=12)
        elif not surf_has[i]:
            fig.text(surf_cx, yc, "n.s.", va="center", ha="center", style="italic", fontsize=12)

    # --- ONE shared diverging T colorbar centred under the two map columns ----------------
    bx0 = gs[0, 1].get_position(fig).x0
    bx1 = gs[0, ncol-1].get_position(fig).x1
    cc = (bx0 + bx1) / 2; wcb = 0.26 * (bx1 - bx0)
    cax = fig.add_axes([cc - wcb / 2, 0.035, wcb, 0.011])
    sm = ScalarMappable(cmap="RdBu_r", norm=Normalize(-VMAX_T, VMAX_T)); sm.set_array([])
    cb = fig.colorbar(sm, cax=cax, orientation="horizontal", extend="both",
                      ticks=[-VMAX_T, 0, VMAX_T])
    _clab = (rf"$T$  ({title}, $\mathbf{{p_{{FDR}}}}$ < 0.05)" if thresholded else r"$T$")
    cb.set_label(_clab, fontsize=11, fontweight="bold")
    cb.ax.tick_params(labelsize=9)
    for t in cb.ax.get_xticklabels():
        t.set_fontweight("bold")

    out = FIG_DIR / outname
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("saved", out)
    return out

## Permutation-based (`FAM_LNM_NOCOV`, `F_PERM_FDR`)

In [ ]:
combined_figure(FAM_LNM_NOCOV, F_PERM_FDR, "Permutation-based",
                "combined_slice_surface_permutation.png")

## Parametric (`FAM_LNM_NOCOV`, `F_PARAM_FDR`)

In [ ]:
combined_figure(FAM_LNM_NOCOV, F_PARAM_FDR, "Parametric",
                "combined_slice_surface_parametric.png")

## One-sample t-test (`FAM_ONESAMPLE`, `F_PARAM_FDR`)

In [ ]:
combined_figure(FAM_ONESAMPLE, F_PARAM_FDR, "One-sample t-test",
                "combined_slice_surface_onesample.png")

## Unthresholded

In [ ]:
combined_figure(FAM_LNM_NOCOV, None, "Unthresholded",
                "combined_slice_surface_unthresholded.png", thresholded=False)

### Unthresholded — slices only

In [ ]:
combined_figure(FAM_LNM_NOCOV, None, "Unthresholded",
                "slices_T_lnm_nocov_unthresholded.png", thresholded=False, surface=False)

## Voxel-based lesion-symptom mapping

In [ ]:
combined_figure(FAM_VLSM, F_PERM_FDR, "VLSM, permutation-based",
                "combined_slice_surface_vlsm_permutation.png")

In [ ]:
combined_figure(FAM_VLSM, None, "VLSM, unthresholded",
                "combined_slice_surface_vlsm_unthresholded.png", thresholded=False)

## Unthresholded T vs beta (slices)

In [ ]:
def slices_T_beta_figure(family, outname=None, keys=None):
    """Unthresholded axial-slice T/beta figure: left = T-statistic, right = beta/COPE on the SAME
    cuts. Slices analogue of surface_render_unthresholded_T_beta (independent robust colour scales,
    radiological orientation, one colorbar per column)."""
    keys = keys if keys is not None else KEYS
    ready = [k for k in keys if has_results(family, k)]
    if not ready:
        print(f"PALM not finished for {family} -- skipping T/beta slices."); return None
    if outname is None:
        outname = f"slices_T_beta_{family}.png"
    def robust(fn):
        allv = np.abs(np.concatenate([load_map(family, k, fn, brain=brain)[brain] for k in ready]))
        v = float(np.round(np.nanpercentile(allv, 97.5), 2)); return v if v > 0 else 1.0
    VMAX_T = robust(F_TSTAT); VMAX_B = robust(F_COPE)
    print(f"[{family}] T scale +/- {VMAX_T} | beta scale +/- {VMAX_B}")
    def vimgs(fn):
        return {k: nib.Nifti1Image(load_map(family, k, fn, brain=brain).astype(np.float32), affine) for k in ready}
    imgT = vimgs(F_TSTAT); imgB = vimgs(F_COPE)
    nrow = len(ready)
    fig = plt.figure(figsize=(16.5, 2.3 * nrow))
    gs = fig.add_gridspec(nrow, 3, width_ratios=[0.18, 1.0, 1.0], wspace=0.02, hspace=0.06,
                          left=0.01, right=0.99, top=0.93, bottom=0.085)
    axT = [fig.add_subplot(gs[i, 1]) for i in range(nrow)]
    axB = [fig.add_subplot(gs[i, 2]) for i in range(nrow)]
    fig.canvas.draw()
    dispT = []
    for i, k in enumerate(ready):
        dT = plotting.plot_stat_map(imgT[k], bg_img=BG, display_mode="z", cut_coords=CUTS, axes=axT[i],
                threshold=1e-6, vmax=VMAX_T, cmap="RdBu_r", colorbar=False, annotate=False,
                black_bg=False, radiological=True); dispT.append(dT)
        plotting.plot_stat_map(imgB[k], bg_img=BG, display_mode="z", cut_coords=CUTS, axes=axB[i],
                threshold=1e-6, vmax=VMAX_B, cmap="RdBu_r", colorbar=False, annotate=False,
                black_bg=False, radiological=True)
    axf = list(dispT[0].axes.values())[0].ax
    axf.text(0.03, 0.97, "R", transform=axf.transAxes, ha="left", va="top", fontsize=9, fontweight="bold")
    axf.text(0.97, 0.97, "L", transform=axf.transAxes, ha="right", va="top", fontsize=9, fontweight="bold")
    def yc(disp, ax):
        try:
            ps=[c.ax.get_position() for c in disp.axes.values()]; return (min(p.y0 for p in ps)+max(p.y1 for p in ps))/2
        except Exception:
            p=ax.get_position(); return (p.y0+p.y1)/2
    ytop = max(c.ax.get_position().y1 for d in dispT for c in d.axes.values())
    for col, head in [(1, "T-statistic"), (2, r"$\beta$ (effect size)")]:
        p = gs[0, col].get_position(fig)
        fig.text((p.x0+p.x1)/2, ytop+0.010, head, fontsize=14, fontweight="bold", ha="center", va="bottom")
    x_lab = gs[0,1].get_position(fig).x0 - 0.006
    for i, k in enumerate(ready):
        fig.text(x_lab, yc(dispT[i], axT[i]), fill(DOMAINS[k], 16), va="center", ha="right", fontsize=12, fontweight="bold")
    def add_cbar(col, vmax, lab):
        p=gs[0,col].get_position(fig); c=(p.x0+p.x1)/2; w=0.5*(p.x1-p.x0)
        cax=fig.add_axes([c-w/2, 0.035, w, 0.011])
        sm=ScalarMappable(cmap="RdBu_r", norm=Normalize(-vmax, vmax)); sm.set_array([])
        cb=fig.colorbar(sm, cax=cax, orientation="horizontal", extend="both", ticks=[-vmax,0,vmax])
        cb.set_label(lab, fontsize=11, fontweight="bold"); cb.ax.tick_params(labelsize=9)
        [t.set_fontweight("bold") for t in cb.ax.get_xticklabels()]
    add_cbar(1, VMAX_T, r"$T$"); add_cbar(2, VMAX_B, r"$\beta$")
    out = FIG_DIR / outname
    fig.savefig(out, dpi=300, bbox_inches="tight"); plt.show(); print("saved", out); return out

In [ ]:
slices_T_beta_figure(FAM_LNM_NOCOV, "slices_T_beta_lnm_nocov.png")